# Deep Learning for NLP – Lab Exercise 1
## Biomedical Text Classification with PyTorch

Multi-label classification of LitCovid articles (BioCreative VII) into 7 topics, with a Bag-of-Words classifier and a CNN.

# Raphaël Thieffry

## Setup

In [ ]:
import random

import numpy as np
import torch as th

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
th.manual_seed(SEED)

device = th.device("cuda" if th.cuda.is_available() else "cpu")
print(f"torch {th.__version__} | device: {device}"
      + (f" ({th.cuda.get_device_name(0)})" if device.type == "cuda" else ""))

ModuleNotFoundError: No module named 'torch'

## 1. Dataset

### 1.1 Load the dataset

In [2]:
from datasets import load_dataset

dataset = load_dataset("KushT/LitCovid_BioCreative")
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['pmid', 'journal', 'title', 'abstract', 'keywords', 'pub_type', 'authors', 'doi', 'label', 'text'],
        num_rows: 24960
    })
    validation: Dataset({
        features: ['pmid', 'journal', 'title', 'abstract', 'keywords', 'pub_type', 'authors', 'doi', 'label', 'text'],
        num_rows: 2500
    })
    test: Dataset({
        features: ['pmid', 'journal', 'title', 'abstract', 'keywords', 'pub_type', 'authors', 'doi', 'label', 'text'],
        num_rows: 6239
    })
})


In [4]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['pmid', 'journal', 'title', 'abstract', 'keywords', 'pub_type', 'authors', 'doi', 'label', 'text'],
        num_rows: 24960
    })
    validation: Dataset({
        features: ['pmid', 'journal', 'title', 'abstract', 'keywords', 'pub_type', 'authors', 'doi', 'label', 'text'],
        num_rows: 2500
    })
    test: Dataset({
        features: ['pmid', 'journal', 'title', 'abstract', 'keywords', 'pub_type', 'authors', 'doi', 'label', 'text'],
        num_rows: 6239
    })
})


**Questions:**

1. How many articles are available in each dataset split?
2. What information is contained in the `text` field?
3. What is the representation of the labels?
4. Can an article have multiple labels?
5. What is the distribution of the seven topics?

### 1 -

| Split | articles |
|-------|-----|
| Train | 24960 | 
| Validation | 2500 |
| test | 6239 |

### 2 -


In [20]:
head5_train_text = dataset["train"][0:5]["text"]
for t in head5_train_text:
    print(t)

[Title]: Potential role for tissue factor in the pathogenesis of hypercoagulability associated with in COVID-19. [Abstract]: In December 2019, a new and highly contagious infectious disease emerged in Wuhan, China. The etiologic agent was identified as a novel coronavirus, now known as Severe Acute Syndrome Coronavirus-2 (SARS-CoV-2). Recent research has revealed that virus entry takes place upon the union of the virus S surface protein with the type I transmembrane metallo-carboxypeptidase, angiotensin converting enzyme 2 (ACE-2) identified on epithelial cells of the host respiratory tract. Virus triggers the synthesis and release of pro-inflammatory cytokines, including IL-6 and TNF-alpha and also promotes downregulation of ACE-2, which promotes a concomitant increase in levels of angiotensin II (AT-II). Both TNF-alpha and AT-II have been implicated in promoting overexpression of tissue factor (TF) in platelets and macrophages. Additionally, the generation of antiphospholipid antibod

It's articles about covid19 in natural language. The articles fit in a structured string according to the corresponding sections (title, abstract etc.).